# Interlude: How to write an MLP in Tensorflow and PyTorch

## Multilayer perceptron
Let's build a multilayer perceptron; MLPs are fully connected, each node in one layer connects with a certain weight to every node in the following layer.

<img src="https://cdn.analyticsvidhya.com/wp-content/uploads/2020/02/ANN-Graph.gif" width="400px"><br>

Try to build one composed by two hidden dense layer with ReLU activation and one dense output layer(units=1) with sigmoid activation.

## Generating Synthetic train and test data



In [1]:
import numpy as np
# Generate dummy data
train_data = np.random.random((1000, 100))
train_labels = np.random.randint(2, size=(1000, 1))
test_data = np.random.random((100, 100))
test_labels = np.random.randint(2, size=(100, 1))



## MLP in Keras

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation

# Create a Sequential
keras_model = Sequential()

# Add two dense layers with 64 and 32 neurons, use relu as activation function
keras_model.add(Dense(64, activation='relu', input_shape=(100,)))
keras_model.add(Dense(32, activation='relu'))

# To produce the output Add a Dense layer with 1 neurons, with sigmoid as activation function
keras_model.add(Dense(1, activation='sigmoid'))

# Compile the model
keras_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


# Train the model, iterating on the data in batches of 32 samples
# The fit function output is a History object. The history.history attribute is a record of
# training loss values and metrics values at successive epochs, as well as validation loss values
# and validation metrics values
epochs=10
batch_size=32
history = keras_model.fit(train_data, train_labels, epochs=epochs, batch_size=batch_size)


# Evaluate
train_loss, train_acc = keras_model.evaluate(train_data, train_labels, verbose=1)
test_loss, test_acc = keras_model.evaluate(test_data, test_labels, verbose=1)
print('Accuracy => Train: %.3f, Test: %.3f' % (train_acc, test_acc))
print('Loss => Train: %.3f, Test: %.3f' % (train_acc, test_acc))

Metal device set to: Apple M1
Epoch 1/10


2025-06-19 14:27:49.048391: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-06-19 14:27:49.048708: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2025-06-19 14:27:49.451790: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


32/32 [==============================] - 2s 27ms/step - loss: 0.6976 - accuracy: 0.5110
Epoch 2/10
32/32 [==============================] - 0s 10ms/step - loss: 0.6902 - accuracy: 0.5210
Epoch 3/10
32/32 [==============================] - 0s 10ms/step - loss: 0.6849 - accuracy: 0.5610
Epoch 4/10
32/32 [==============================] - 0s 10ms/step - loss: 0.6794 - accuracy: 0.5850
Epoch 5/10
32/32 [==============================] - 0s 9ms/step - loss: 0.6721 - accuracy: 0.5890
Epoch 6/10
24/32 [=====================>........] - ETA: 0s - loss: 0.6659 - accuracy: 0.6146

## MPL in Tensorflow

In [ ]:
import tensorflow as tf


# Convert to tensors
train_X_tf = tf.convert_to_tensor(train_data, dtype=tf.float32)
train_y_tf = tf.convert_to_tensor(train_labels, dtype=tf.float32)
test_X_tf = tf.convert_to_tensor(test_data, dtype=tf.float32)
test_y_tf = tf.convert_to_tensor(test_labels, dtype=tf.float32)


# Define model
class TFMLP(tf.Module):
    def __init__(self):
        super().__init__()
        self.w1 = tf.Variable(tf.random.normal([100, 64]), name='w1')
        self.b1 = tf.Variable(tf.zeros([64]), name='b1')
        self.w2 = tf.Variable(tf.random.normal([64, 32]), name='w2')
        self.b2 = tf.Variable(tf.zeros([32]), name='b2')
        self.w3 = tf.Variable(tf.random.normal([32, 1]), name='w3')
        self.b3 = tf.Variable(tf.zeros([1]), name='b3')

    def __call__(self, x):
        x = tf.nn.relu(tf.matmul(x, self.w1) + self.b1)
        x = tf.nn.relu(tf.matmul(x, self.w2) + self.b2)
        x = tf.sigmoid(tf.matmul(x, self.w3) + self.b3)
        return x



# Train loop
tf_model = TFMLP()
optimizer = tf.optimizers.Adam()
loss_fn = tf.keras.losses.BinaryCrossentropy()

batch_size = 32
epochs=10
train_ds = tf.data.Dataset.from_tensor_slices((train_X_tf, train_y_tf)).batch(batch_size)

for epoch in range(epochs):
    for x_batch, y_batch in train_ds:
        with tf.GradientTape() as tape:
            preds = tf_model(x_batch)
            loss = loss_fn(y_batch, preds)
        grads = tape.gradient(loss, tf_model.trainable_variables)
        optimizer.apply_gradients(zip(grads, tf_model.trainable_variables))
    print(f"Epoch {epoch+1}: Loss = {loss.numpy():.4f}")

# Test loop (batched)
test_ds = tf.data.Dataset.from_tensor_slices((test_X_tf, test_y_tf)).batch(batch_size)
correct = 0
total = 0
for x_batch, y_batch in test_ds:
    preds = tf_model(x_batch)
    predicted = tf.cast(preds > 0.5, tf.float32)
    correct += tf.reduce_sum(tf.cast(predicted == y_batch, tf.float32)).numpy()
    total += y_batch.shape[0]
print(f"TF Test Accuracy: {correct / total:.4f}")


## MPL in PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Convert to tensors
train_X_pt = torch.tensor(train_data, dtype=torch.float32)
train_y_pt = torch.tensor(train_labels, dtype=torch.float32)
test_X_pt = torch.tensor(test_data, dtype=torch.float32)
test_y_pt = torch.tensor(test_labels, dtype=torch.float32)



batch_size = 32
epochs=10


train_loader = DataLoader(TensorDataset(train_X_pt, train_y_pt), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(TensorDataset(test_X_pt, test_y_pt), batch_size=batch_size, shuffle=False)

# Define model
class TorchMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(100, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.sigmoid(self.fc3(x))
        return x

torch_model = TorchMLP()
criterion = nn.BCELoss()
optimizer = optim.Adam(torch_model.parameters())

# Training
for epoch in range(epochs):
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = torch_model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}: Loss = {loss.item():.4f}")

# Batched test loop
correct = 0
total = 0
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        outputs = torch_model(batch_X)
        preds = (outputs > 0.5).float()
        correct += (preds == batch_y).sum().item()
        total += batch_y.size(0)
print(f"PyTorch Test Accuracy: {correct / total:.4f}")
